# MedTrack DV — Full Cleaning Pipeline (Admissions + Resources)
Cleans **two** raw sources and merges them into one analysis-ready dataset for the
Power BI dashboard:

1. `hospital_raw_data.csv` — messy patient admission records
2. `hospital_department_resources_raw.csv` — messy staffing & equipment data (new)

Output: `hospital_cleaned_extended.csv` — same admissions data as before, now with
`Staff_Count` and `Equipment_Count` added per row.

## 1. Load both raw files
Upload both CSVs to the Colab file browser first (left sidebar → Files → upload).

In [ ]:
import pandas as pd
import numpy as np
import re

raw_adm = pd.read_csv('hospital_raw_data.csv')
raw_res = pd.read_csv('hospital_department_resources_raw.csv')
print('Raw admissions:', raw_adm.shape)
print('Raw resources:', raw_res.shape)
raw_res.head()

## 2. Clean the admissions data
Same process as the original cleaning notebook — drop junk, standardize text, fix dates/booleans/numbers, strip ID prefixes, dedupe.

In [ ]:
df = raw_adm.copy()
df = df.drop(columns=['Unnamed: 0'])
df = df.dropna(how='all')

text_cols = ['Hospital','Department','Region','Patient_Type','Gender']
for c in text_cols:
    df[c] = df[c].astype(str).str.strip()
    df[c] = df[c].str.replace(r'\s+', ' ', regex=True)

hospital_map = {
    'city care hospital': 'City Care Hospital', 'green valley hospital': 'Green Valley Hospital',
    'sunrise medical center': 'Sunrise Medical Center', 'metro health institute': 'Metro Health Institute',
    'healthplus hospital': 'HealthPlus Hospital',
}
dept_map = {
    'general medicine': 'General Medicine', 'gen medicine': 'General Medicine', 'surgery': 'Surgery',
    'pediatrics': 'Pediatrics', 'peds': 'Pediatrics', 'orthopedics': 'Orthopedics', 'ortho': 'Orthopedics',
    'cardiology': 'Cardiology', 'emergency': 'Emergency', 'er': 'Emergency',
    'icu': 'ICU', 'i.c.u': 'ICU', 'i.c.u.': 'ICU',
}
patient_type_map = {
    'inpatient': 'Inpatient', 'in-patient': 'Inpatient', 'outpatient': 'Outpatient', 'out-patient': 'Outpatient',
    'emergency': 'Emergency', 'day care': 'Day Care', 'daycare': 'Day Care',
}
gender_map = {'male': 'Male', 'm': 'Male', 'female': 'Female', 'f': 'Female'}

def canon(series, mapping):
    return series.str.lower().map(mapping).fillna(series)

df['Hospital'] = canon(df['Hospital'], hospital_map)
df['Department'] = canon(df['Department'], dept_map)
df['Patient_Type'] = canon(df['Patient_Type'], patient_type_map)
df['Gender'] = canon(df['Gender'], gender_map)
df['Region'] = df['Region'].str.lower().str.title()

In [ ]:
def to_bool(v):
    if isinstance(v, bool):
        return v
    return str(v).strip().lower() in ('true', 'yes', '1')

df['Readmitted_30Days'] = df['Readmitted_30Days'].apply(to_bool)
df['Admission_Date'] = pd.to_datetime(df['Admission_Date'], format='mixed')
df['Discharge_Date'] = pd.to_datetime(df['Discharge_Date'], format='mixed')

def extract_number(v):
    if pd.isna(v):
        return np.nan
    m = re.search(r'-?\d+\.?\d*', str(v))
    return float(m.group()) if m else np.nan

df['Length_of_Stay_Days'] = df['Length_of_Stay_Days'].apply(extract_number).astype(int)
df['Total_Beds'] = df['Total_Beds'].apply(extract_number).astype(int)
df['Patient_Age'] = df['Patient_Age'].apply(extract_number).astype(int)
df['Admission_ID'] = df['Admission_ID'].astype(str).str.replace('ADM-', '', regex=False).astype(int)
df['Patient_ID'] = df['Patient_ID'].astype(str).str.replace('PT-', '', regex=False).astype(int)
df = df.drop_duplicates()

df['Year'] = df['Admission_Date'].dt.year
df['Month_Num'] = df['Admission_Date'].dt.month
df['Month_Name'] = df['Admission_Date'].dt.strftime('%b')

col_order = ['Admission_ID','Patient_ID','Admission_Date','Discharge_Date','Hospital','Department',
             'Region','Patient_Type','Length_of_Stay_Days','Readmitted_30Days','Patient_Age','Gender',
             'Year','Month_Num','Month_Name','Total_Beds']
df = df[col_order].sort_values('Admission_ID').reset_index(drop=True)
print('Cleaned admissions:', df.shape)

## 3. Clean the NEW resources data (Staff & Equipment)
Same text-standardization logic reused from step 2 — this is the *new* source being added.

In [ ]:
res = raw_res.copy()
for c in ['Hospital','Department']:
    res[c] = res[c].astype(str).str.strip().str.replace(r'\s+', ' ', regex=True)

res['Hospital'] = canon(res['Hospital'], hospital_map)
res['Department'] = canon(res['Department'], dept_map)

res['Staff_Count'] = res['Staff_Count'].apply(extract_number).astype(int)
res['Equipment_Count'] = res['Equipment_Count'].apply(extract_number).astype(int)

res = res.drop_duplicates(subset=['Hospital','Department'])
res = res.sort_values(['Hospital','Department']).reset_index(drop=True)
print('Cleaned resources:', res.shape)
res.head()

## 4. Merge the two cleaned sources
Every admission row gets its department's `Staff_Count` and `Equipment_Count` joined in
on `Hospital` + `Department` — this is the 'add extra data' step.

In [ ]:
final = df.merge(res, on=['Hospital', 'Department'], how='left')

unmatched = final['Staff_Count'].isnull().sum()
print('Final shape:', final.shape)
print('Rows that failed to match a resource record:', unmatched)
final.head()

## 5. Save the final enriched dataset

In [ ]:
final.to_csv('hospital_cleaned_extended.csv', index=False)
print('Saved hospital_cleaned_extended.csv —', final.shape[0], 'rows,', final.shape[1], 'columns')

## 6. (Optional) Verify against the known-clean reference
If you also have `hospital_admissions_dataset.csv` and `hospital_department_resources.csv`
handy, this confirms the cleaning reproduced them exactly.

In [ ]:
# truth_adm = pd.read_csv('hospital_admissions_dataset.csv')
# truth_res = pd.read_csv('hospital_department_resources.csv')
# truth_final = truth_adm.merge(truth_res, on=['Hospital','Department'], how='left')
# a = final.sort_values('Admission_ID').reset_index(drop=True)
# b = truth_final.sort_values('Admission_ID').reset_index(drop=True)
# for c in a.columns:
#     print(c, (a[c].astype(str) == b[c].astype(str)).all())